In [ ]:
import h5py 
import numpy as np
import math 
import cv2
import tqdm 

In [ ]:
h5py_paths = [
    "/home/ns1254/data_franka/3task_eggplant_99/pickup_eggplant_33_t1b3_march25.hdf5",
    "/home/ns1254/data_franka/3task_eggplant_99/place2drawer_33_t2b3_march25.hdf5",
    "/home/ns1254/data_franka/3task_eggplant_99/closedrawer_33_t3b3_march25.hdf5"
]
language_instructions = [
    "pick up the eggplant",
    "place the grasped object into the drawer",
    "close the drawer"
]


REPO_NAME = "nsojib/eggplant_pickup_place_close_drawer_t3_99" 

In [ ]:
# https://github.com/ARISE-Initiative/robosuite/blob/eafb81f54ffc104f905ee48a16bb15f059176ad3/robosuite/utils/transform_utils.py

def mat2quat(rmat):
    """
    Converts given rotation matrix to quaternion.

    Args:
        rmat (np.array): 3x3 rotation matrix

    Returns:
        np.array: (x,y,z,w) float quaternion angles
    """
    M = np.asarray(rmat).astype(np.float32)[:3, :3]

    m00 = M[0, 0]
    m01 = M[0, 1]
    m02 = M[0, 2]
    m10 = M[1, 0]
    m11 = M[1, 1]
    m12 = M[1, 2]
    m20 = M[2, 0]
    m21 = M[2, 1]
    m22 = M[2, 2]
    # symmetric matrix K
    K = np.array(
        [
            [m00 - m11 - m22, np.float32(0.0), np.float32(0.0), np.float32(0.0)],
            [m01 + m10, m11 - m00 - m22, np.float32(0.0), np.float32(0.0)],
            [m02 + m20, m12 + m21, m22 - m00 - m11, np.float32(0.0)],
            [m21 - m12, m02 - m20, m10 - m01, m00 + m11 + m22],
        ]
    )
    K /= 3.0
    # quaternion is Eigen vector of K that corresponds to largest eigenvalue
    w, V = np.linalg.eigh(K)
    inds = np.array([3, 0, 1, 2])
    q1 = V[inds, np.argmax(w)]
    if q1[0] < 0.0:
        np.negative(q1, q1)
    inds = np.array([1, 2, 3, 0])
    return q1[inds]
    
def mat2pose(hmat):
    """
    Converts a homogeneous 4x4 matrix into pose.

    Args:
        hmat (np.array): a 4x4 homogeneous matrix

    Returns:
        2-tuple:

            - (np.array) (x,y,z) position array in cartesian coordinates
            - (np.array) (x,y,z,w) orientation array in quaternion form
    """
    pos = hmat[:3, 3]
    orn = mat2quat(hmat[:3, :3])
    return pos, orn
    
def mat4(array):
    """
    Converts an array to 4x4 matrix.

    Args:
        array (n-array): the array in form of vec, list, or tuple

    Returns:
        np.array: a 4x4 numpy matrix
    """
    return np.array(array, dtype=np.float32).reshape((4, 4))


def quat2axisangle(quat):
    """
    Converts quaternion to axis-angle format.
    Returns a unit vector direction scaled by its angle in radians.

    Args:
        quat (np.array): (x,y,z,w) vec4 float angles

    Returns:
        np.array: (ax,ay,az) axis-angle exponential coordinates
    """
    # clip quaternion
    if quat[3] > 1.0:
        quat[3] = 1.0
    elif quat[3] < -1.0:
        quat[3] = -1.0

    den = np.sqrt(1.0 - quat[3] * quat[3])
    if math.isclose(den, 0.0):
        # This is (close to) a zero degree rotation, immediately return
        return np.zeros(3)

    return (quat[:3] * 2.0 * math.acos(quat[3])) / den

In [ ]:
def franka_ee_states_to_pos_quat(ee_states):
    # ee_states = demo_franka['obs']['ee_states'][:]  # (N, 16)
    # T_all = np.array([mat4(ee_states[i]) for i in range(ee_states.shape[0])])  # (N, 4, 4)
    T_all = ee_states.reshape(-1, 4, 4, order='F')  # (N, 4, 4)
    pos_all=[]
    ori_all=[]
    for i in range(T_all.shape[0]):
        pos, orn = mat2pose(T_all[i])
        pos_all.append(pos)
        ori_all.append(orn)
    pos_all = np.array(pos_all)
    quat_all = np.array(ori_all)
    return pos_all, quat_all

In [ ]:
def franka_obs_to_libero_like_obs(franka_obs):
    joint_states = franka_obs['joint_states'][:]  # (N, 7)
    gripper_states = franka_obs['gripper_states'][:]  # (N, 1) 
    gripper_states = np.repeat(gripper_states, repeats=2, axis=1)  # (N, 2)

    # ee_states from 16d vector to pos + rpy
    ee_states = franka_obs['ee_states'][:]  # (N, 16)
    pos_all, quat_all = franka_ee_states_to_pos_quat(ee_states) 
    rpy_all = np.array([quat2axisangle(quat_all[i]) for i in range(quat_all.shape[0])]) 
    ee_states = np.hstack([pos_all, rpy_all] )

    # convert images to 256x256
    img = franka_obs['agentview_rgb'][:]  
    img_wrist = franka_obs['eye_in_hand_rgb'][:]
    img = np.array([cv2.resize(img[i], (256,256)) for i in range(img.shape[0])])
    img_wrist = np.array([cv2.resize(img_wrist[i], (256,256)) for i in range(img_wrist.shape[0])])

    libero_like_obs = {
        'ee_states': ee_states,  # (N, 6)
        'ee_pos': ee_states[:, :3],  # (N, 3)
        'ee_ori': ee_states[:, 3:],  # (N, 3)
        'gripper_states': gripper_states,  # (N, 2)
        'joint_states': joint_states,  # (N, 7)
        'agentview_rgb': img,  # (N, 256, 256, 3)
        'eye_in_hand_rgb': img_wrist,  # (N, 256, 256, 3)
    }
    return libero_like_obs
    

In [ ]:
import shutil

# from lerobot.common.datasets.lerobot_dataset import LEROBOT_HOME
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset

In [ ]:
# Create LeRobot dataset, define features to store
# OpenPi assumes that proprio is stored in `state` and actions in `action`
# LeRobot assumes that dtype of image data is `image`
dataset = LeRobotDataset.create(
    repo_id=REPO_NAME,
    robot_type="panda",
    fps=20,
    features={
        "image": {
            "dtype": "image",
            "shape": (256, 256, 3),
            "names": ["height", "width", "channel"],
        },
        "wrist_image": {
            "dtype": "image",
            "shape": (256, 256, 3),
            "names": ["height", "width", "channel"],
        },
        "state": {
            "dtype": "float32",
            "shape": (8,),
            "names": ["state"],
        },
        "actions": {
            "dtype": "float32",
            "shape": (7,),
            "names": ["actions"],
        },
    },
    image_writer_threads=10,
    image_writer_processes=5,
)

In [ ]:
print(f"Created dataset with repo_id: {dataset.repo_id}")
print(f'Total number of files to process: {len(language_instructions)}')

for i in range(len(language_instructions)):
    file_path = h5py_paths[i]
    file_franka = h5py.File(file_path, 'r')
    prompt = language_instructions[i]

    demo_names = list(file_franka['data'].keys()) 
    demo_names = sorted(demo_names, key=lambda x: int(x.split('_')[-1]))

    print(f"\nProcessing file {file_path} with prompt: {prompt} and {len(demo_names)} demos")

    for i, demo_name in enumerate(demo_names):

        demo_franka=file_franka[f'data/{demo_name}']

        actions = demo_franka['actions'][:]
        libero_like_obs = franka_obs_to_libero_like_obs(demo_franka['obs'])

        for i in range(actions.shape[0]):
            frame = {
                "image": libero_like_obs['agentview_rgb'][i],
                "wrist_image": libero_like_obs['eye_in_hand_rgb'][i],
                "state": np.hstack([
                    libero_like_obs['ee_states'][i],
                    libero_like_obs['gripper_states'][i],
                ]).astype(np.float32),
                "actions": actions[i].astype(np.float32),
                'task': prompt
            }
            dataset.add_frame(frame)
        dataset.save_episode()

    # dataset.consolidate(run_compute_stats=False)

print(f"\nDataset creation complete! Dataset repo_id: {dataset.repo_id}")